In [ ]:
# ===================== CHUNK 1: CLUSTERING GAMBAR =====================
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. LOAD DATA (folder gambar, boleh ada subfolder per kelas)
paths = glob.glob("dataset/**/*.*", recursive=True)                    # GANTI nama folder
paths = [p for p in paths if p.lower().endswith((".jpg", ".jpeg", ".png"))]

images, labels = [], []
for p in paths:
    img = Image.open(p).convert("RGB").resize((64, 64))                 # samakan warna & ukuran
    images.append(np.array(img))
    labels.append(os.path.basename(os.path.dirname(p)))                 # nama folder = label
images = np.array(images)
print("Shape data:", images.shape)                                      # (jumlah, 64, 64, 3)

# 2. EDA
print(pd.Series(labels).value_counts())

fig, ax = plt.subplots(1, 5, figsize=(12, 3))
for i in range(5):
    ax[i].imshow(images[i]); ax[i].set_title(labels[i]); ax[i].axis("off")
plt.show()

plt.hist(images.ravel(), bins=50)
plt.title("Distribusi nilai piksel"); plt.show()

# 3. PREPROCESSING: flatten -> normalisasi -> standarisasi -> PCA
X = images.reshape(len(images), -1) / 255.0                             # gambar jadi 1 baris angka 0-1
X = StandardScaler().fit_transform(X)
X = PCA(n_components=min(50, len(X))).fit_transform(X)                  # kecilkan dimensi
print("Shape setelah PCA:", X.shape)

# 4. ELBOW METHOD + SILHOUETTE
K = range(2, 11)
inertia, sil = [], []
for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X, km.labels_))

plt.plot(K, inertia, "o-"); plt.title("Elbow Method"); plt.xlabel("k"); plt.ylabel("Inertia"); plt.show()
plt.plot(K, sil, "o-"); plt.title("Silhouette Score"); plt.xlabel("k"); plt.show()
print(pd.DataFrame({"k": K, "inertia": inertia, "silhouette": sil}))

# 5. KMEANS + EVALUASI
# 5a. k langsung (dari soal)
k = 3                                                                   # GANTI sesuai soal
km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
print("k langsung =", k, "| Inertia:", km.inertia_, "| Silhouette:", silhouette_score(X, km.labels_))

# 5b. k dari elbow (lihat siku di grafik)
k = 4                                                                   # GANTI sesuai siku grafik elbow
km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
print("k elbow    =", k, "| Inertia:", km.inertia_, "| Silhouette:", silhouette_score(X, km.labels_))
cluster = km.labels_

# 6. VISUALISASI CLUSTER (PCA 2D)
X2 = PCA(n_components=2).fit_transform(X)
plt.scatter(X2[:, 0], X2[:, 1], c=cluster, cmap="tab10")
plt.title(f"Hasil KMeans (k={k})"); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.colorbar(label="cluster"); plt.show()

# 7. INSIGHT: jumlah, contoh gambar tiap cluster, cek ke label asli
print(pd.Series(cluster).value_counts())

for c in range(k):
    idx = np.where(cluster == c)[0][:5]
    fig, ax = plt.subplots(1, 5, figsize=(10, 2))
    for j in range(5):
        ax[j].axis("off")
        if j < len(idx):
            ax[j].imshow(images[idx[j]])
    plt.suptitle(f"Cluster {c}"); plt.show()

print(pd.crosstab(cluster, np.array(labels)))                           # kalau ada label asli

In [ ]:
# ===================== CHUNK 2: CLUSTERING TEKS =====================
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from collections import Counter
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. LOAD DATA
df = pd.read_csv("data.csv")                                            # GANTI nama file
kolom = "text"                                                          # GANTI nama kolom teks
df = df.dropna(subset=[kolom]).drop_duplicates(subset=[kolom])
print("Shape data:", df.shape)
print(df.head())

# 2. EDA
df["jumlah_kata"] = df[kolom].str.split().str.len()
print(df["jumlah_kata"].describe())
plt.hist(df["jumlah_kata"], bins=30); plt.title("Distribusi panjang teks"); plt.show()

# 3. PREPROCESSING: lowercase -> hapus link/mention/simbol/angka -> tokenisasi -> stopword
nltk.download("stopwords")
stop = set(stopwords.words("indonesian"))                               # "english" kalau data Inggris
stop.update(["yg", "ga", "gak", "aja", "banget", "dgn", "utk", "sm", "tp", "jd", "gue", "nya", "kalo", "tdk"])   # [GANTI]

def clean(text):
    text = text.lower()
    text = re.sub(r"http\S+|@\w+", " ", text)                           # hapus link & mention
    text = re.sub(r"[^a-z\s]", " ", text)                               # hapus angka & simbol
    words = text.split()                                                # tokenisasi
    words = [w for w in words if w not in stop]                         # hapus stopword
    return " ".join(words)


df["clean"] = df[kolom].apply(clean)
print(df[[kolom, "clean"]].head())

# EDA setelah prepro: kata terbanyak + wordcloud
top = Counter(" ".join(df["clean"]).split()).most_common(15)
pd.DataFrame(top, columns=["kata", "jumlah"]).plot.barh(x="kata", y="jumlah"); plt.show()

wc = WordCloud(background_color="white").generate(" ".join(df["clean"]))
plt.imshow(wc); plt.axis("off"); plt.show()

# 4. TF-IDF (teks -> angka)
tfidf = TfidfVectorizer(max_features=1000)
X = tfidf.fit_transform(df["clean"])
print("Shape TF-IDF:", X.shape)

# 5. ELBOW METHOD + SILHOUETTE
K = range(2, 11)
inertia, sil = [], []
for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X, km.labels_))

plt.plot(K, inertia, "o-"); plt.title("Elbow Method"); plt.xlabel("k"); plt.ylabel("Inertia"); plt.show()
plt.plot(K, sil, "o-"); plt.title("Silhouette Score"); plt.xlabel("k"); plt.show()
print(pd.DataFrame({"k": K, "inertia": inertia, "silhouette": sil}))

# 6. KMEANS + EVALUASI
# 6a. k langsung (dari soal)
k = 3                                                                   # GANTI sesuai soal
km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
print("k langsung =", k, "| Inertia:", km.inertia_, "| Silhouette:", silhouette_score(X, km.labels_))

# 6b. k dari elbow (lihat siku di grafik)
k = 4                                                                   # GANTI sesuai siku grafik elbow
km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
print("k elbow    =", k, "| Inertia:", km.inertia_, "| Silhouette:", silhouette_score(X, km.labels_))
df["cluster"] = km.labels_

# 7. VISUALISASI (PCA 2D) + INSIGHT
X2 = PCA(n_components=2).fit_transform(X.toarray())
plt.scatter(X2[:, 0], X2[:, 1], c=df["cluster"], cmap="tab10")
plt.title(f"Hasil KMeans (k={k})"); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.colorbar(label="cluster"); plt.show()

print(df["cluster"].value_counts())

terms = tfidf.get_feature_names_out()
for c in range(k):
    top_idx = km.cluster_centers_[c].argsort()[::-1][:10]               # 10 kata terpenting
    print(f"\nCluster {c}:", ", ".join(terms[top_idx]))
    for teks in df[df["cluster"] == c][kolom].head(3):                  # contoh teks
        print("  -", teks)